# CADICA individual lesion detector
Upload `CADICA_individual_training_bundle.zip` to the Colab Files panel at `/content/`. Select a GPU runtime. No CT or other X-ray dataset is mixed in. Source boxes/classes are preserved. Model: pretrained Faster R-CNN ResNet50 FPN. Experimental patient split, validation mAP selection, early stopping and LR reduction; test untouched. CC BY 4.0: https://data.mendeley.com/datasets/p9bpx9ctcv/5
This notebook is prepared, not evidence of completed training. Save checkpoints before runtime expiry.

In [ ]:
import pathlib, zipfile, hashlib, subprocess, sys
archive=pathlib.Path('/content/CADICA_individual_training_bundle.zip')
assert archive.is_file(), 'Upload the prepared bundle using Colab Files first'
h=hashlib.sha256()
with archive.open('rb') as f:
    for block in iter(lambda:f.read(1024*1024),b''): h.update(block)
assert h.hexdigest()=='142067ea750fc0fe6ac57f03a6a03f74e35edcec3c2bdffc8b6dc50ea1648c3f', 'Bundle integrity failed'
PROJECT=pathlib.Path('/content/cadica_individual'); PROJECT.mkdir(exist_ok=True)
with zipfile.ZipFile(archive) as z:
    assert z.testzip() is None
    assert all((PROJECT/n).resolve().is_relative_to(PROJECT.resolve()) for n in z.namelist())
    z.extractall(PROJECT)
subprocess.run([sys.executable,'-m','pip','install','-q','torchmetrics[detection]','pycocotools'],check=True)
import torch
assert torch.cuda.is_available(), 'Select GPU runtime'
print(torch.cuda.get_device_name(0))


In [ ]:
subprocess.run([sys.executable,'scripts/train_cadica_detector.py','--epochs','40','--patience','8','--batch-size','2'],cwd=PROJECT,check=True)


In [ ]:
import shutil
from google.colab import files
result=shutil.make_archive('/content/CADICA_results','zip',PROJECT,'models/individual/cadica')
files.download(result)
